# 04 · Por Trás dos Panos (Caso A)

**Teoria**: docs/02-rdds-lineage-partitions.md, docs/03-transformations-actions-dag.md, docs/04-dataframes-catalyst-tungsten.md, docs/06-persistence-and-optimization.md

Você já confia nos resultados dos notebooks 01-03. Agora: por que o Spark
levou aquele tempo pra calcular, e como ele decide o plano de execução?

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

spark = get_local_session("04-por-tras-dos-panos")

## Lazy evaluation e lineage de RDD

Nada abaixo executa até uma **ação** (`collect`, `sum`, ...) ser chamada.
Transformações (`filter`, `map`) só ficam registradas no grafo de
lineage.

In [ ]:
sc = spark.sparkContext

numbers = sc.parallelize(range(1, 1_000_001), numSlices=8)
print(f"Partições: {numbers.getNumPartitions()}")

# Transformações — preguiçosas, nada roda ainda
evens = numbers.filter(lambda n: n % 2 == 0)
squared = evens.map(lambda n: n * n)

# Ação — É AQUI que o Spark de fato constrói o DAG e executa
total = squared.sum()
print(f"Soma dos quadrados dos pares de 1..1.000.000: {total:,}")

In [ ]:
# O grafo de lineage que o Spark usaria pra recomputar este RDD em caso de falha:
print(squared.toDebugString().decode())

## RDD vs. DataFrame: o plano que o Catalyst enxerga

Mesma lógica, mas o DataFrame dá ao Spark um schema — é isso que torna a
otimização do Catalyst possível. Compare o `.explain()` abaixo contra o
lineage opaco de RDD acima.

In [ ]:
df = spark.range(1, 1_000_001).toDF("n")
resultado = df.filter(df.n % 2 == 0).selectExpr("sum(n * n) as soma_quadrados")
resultado.explain(True)
resultado.show()

## O Catalyst nos joins do notebook 03

`BroadcastHashJoin` (sem Shuffle da tabela grande) vs. `SortMergeJoin`
(Shuffle dos dois lados) — o mesmo par de junções do notebook 03, agora
olhando o plano por trás delas.

In [ ]:
from pyspark.sql.functions import broadcast

vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))

join_broadcast = vendas.join(broadcast(empresas), "id_empresa")
print("--- broadcast join (empresas, 50 linhas) ---")
join_broadcast.explain()

join_shuffle = vendas.join(funcionarios, "id_funcionario")
print("--- shuffle join (funcionarios, milhares de linhas) ---")
join_shuffle.explain()

## Spark SQL — mesmo motor, sintaxe SQL

In [ ]:
vendas.createOrReplaceTempView("vendas")
empresas.createOrReplaceTempView("empresas")
funcionarios.createOrReplaceTempView("funcionarios")

resultado_sql = spark.sql("""
    SELECT e.setor, SUM(v.valor) AS total_vendas
    FROM vendas v
    JOIN funcionarios f ON v.id_funcionario = f.id_funcionario
    JOIN empresas e ON f.id_empresa = e.id_empresa
    GROUP BY e.setor
    ORDER BY total_vendas DESC
    LIMIT 10
""")
resultado_sql.show()

## Cache: pagar uma vez, reusar muitas

In [ ]:
import time

from pyspark.sql.functions import col

vendas_filtradas = vendas.filter(col("valor") > 50)

start = time.perf_counter()
contagem_1 = vendas_filtradas.count()
sem_cache_segundos = time.perf_counter() - start

vendas_filtradas.cache()
vendas_filtradas.count()  # materializa o cache

start = time.perf_counter()
contagem_2 = vendas_filtradas.groupBy("ano").count().count()
com_cache_segundos = time.perf_counter() - start

print(f"Primeiro count() (sem cache ainda): {sem_cache_segundos:.3f}s")
print(f"groupBy sobre dado cacheado:         {com_cache_segundos:.3f}s")
vendas_filtradas.unpersist()

## PySpark vs. Pandas nesta escala

Em algumas centenas de milhares de linhas, o Pandas não paga overhead de
distribuição/serialização — é bem provável que vença aqui. A lição não é
"Spark é lento", é "use a ferramenta certa pro tamanho do dado" (ver
docs/05).

In [ ]:
pdf = vendas.toPandas()

start = time.perf_counter()
resultado_pandas = pdf.groupby("ano")["valor"].sum().sort_values(ascending=False)
pandas_segundos = time.perf_counter() - start

start = time.perf_counter()
resultado_spark = vendas.groupBy("ano").agg({"valor": "sum"}).collect()
spark_segundos = time.perf_counter() - start

print(f"Pandas groupBy: {pandas_segundos:.4f}s")
print(f"Spark groupBy:  {spark_segundos:.4f}s")
print()
print("Nesta escala, o menor overhead geralmente vence — o Lab 13 revisita")
print("essa mesma comparação numa escala onde a história do Spark muda.")

In [ ]:
spark.stop()